In [ ]:
! pip install -q transformers datasets evaluate accelerate huggingface_hub

Causal language modeling predicts the next token in a sequence of tokens, and the model can only attend to tokens on
the left. This means the model cannot see future tokens.

This guide will show you how to:
1. Finetune [DistilGPT2](https://huggingface.co/distilbert/distilgpt2) on the [r/askscience](https://www.reddit.com/r/askscience/) subset of the [ELI5](https://huggingface.co/datasets/dany0407/eli5_category) dataset.
2. Use your finetuned model for inference.

## Load dependencies

In [ ]:
import os, math
from datasets import load_dataset
from transformers import (
pipeline, AutoTokenizer, DataCollatorForLanguageModeling,
AutoModelForCausalLM, TrainingArguments, Trainer
)
from huggingface_hub import notebook_login

## Config

In [ ]:
user_name = "amin-oj"
ckpt_name = "my_first_eli5_clm-model"
dataset_name = "dany0407/eli5_category"
model_name = "distilbert/distilgpt2"
push_to_hub=True

## HF Login

In [ ]:
notebook_login()

## Load dataset

In [ ]:
eli5 = load_dataset(dataset_name, split="train[:5000]")
eli5 = eli5.train_test_split(test_size=0.2)
print(f"initial data sample\n{eli5["train"][0]}")
eli5 = eli5.flatten()
print(f"data sample after flattenening\n{eli5["train"][0]}")

- While this may look like a lot, you're only really interested in the `text` field.
- What's cool about language modeling tasks is you don't need labels (also known as an unsupervised task) because the next word *is* the label.
- After flattening, Each subfield is a separate column as indicated by the `answers` prefix, and the `text` field is a list now.
    - Instead of tokenizing each sentence separately, convert the list to a string so you can jointly tokenize them.

## Preprocess

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    return tokenizer([" ".join(x) for x in examples["answers.text"]])
    # no padding no truncation here

- To apply this preprocessing function over the entire dataset, use the 🤗 Datasets [map](https://huggingface.co/docs/datasets/main/en/package_reference/main_classes#datasets.Dataset.map) method.
- You can speed up the `map` function by
    - setting `batched=True` to process multiple elements of the dataset at once,
    - and increasing the number of processes with `num_proc`.
- Remove any columns you don't need.

In [ ]:
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=2,
    remove_columns=eli5["train"].column_names,
)

This dataset contains the token sequences, but some of these are longer than the maximum input length for the model.

You can now use a second preprocessing function to

- concatenate all the sequences
- split the concatenated sequences into shorter chunks defined by `block_size`, which should be both shorter than the maximum input length and short enough for your GPU RAM.

In [ ]:
def group_texts(examples, block_size = 128):
    pid = os.getpid()
    # print(type(examples)) # lazyBatch
    split_keys = list(examples.keys()) # ['input_ids', 'attention_mask']
    print(f"total length of the initial batch for pid-{pid}: {len(examples[split_keys[0]])}")
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in split_keys}
    total_length = len(concatenated_examples[split_keys[0]])
    print(f"total length of the concatenated batch for pid-{pid}: {total_length}")

    
    # We drop the small remainder; we could add padding if
    # the model supported it instead of this drop. You can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy() # for CLM, label is input seq shifted to the right.
    return result

# this is applied on both train and test datasets separately
lm_dataset = tokenized_eli5.map(group_texts, batched=True, num_proc=2)

In [ ]:
print(lm_dataset['train'])
print(tokenized_eli5['train'])

## Train

- Now create a batch of examples using [DataCollatorForLanguageModeling](https://huggingface.co/docs/transformers/main/en/main_classes/data_collator#transformers.DataCollatorForLanguageModeling).
- It's more efficient to *dynamically pad* the sentences to the longest length in a batch during collation
    - instead of padding the whole dataset to the maximum length.
- Do we need dyanamic padding here? all sequences are now of `block_size` length.
- Use the end-of-sequence token as the padding token and set `mlm=False`.
    - This will use the inputs as labels shifted to the right by one element:

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
model = AutoModelForCausalLM.from_pretrained(model_name)

At this point, only three steps remain:

1. Define your training hyperparameters in [TrainingArguments](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments).
    2. The only required parameter is `output_dir` which specifies where to save your model.
    3. You'll push this model to the Hub by setting `push_to_hub=True`
        4. (you need to be signed in to Hugging Face to upload your model).
3. Pass the training arguments to [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer) along with the model, datasets, and data collator.
4. Call [train()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.train) to finetune your model.

In [ ]:
training_args = TrainingArguments(
    output_dir=ckpt_name,
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=push_to_hub,
    report_to="none" # to disable wandb login
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()

- Once training is completed, use the [evaluate()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.evaluate) method to evaluate your model and get its perplexity
- Then share your model to the Hub with the [push_to_hub()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.push_to_hub) method so everyone can use your model:

In [ ]:
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")
if push_to_hub: trainer.push_to_hub()

## Inference

### Simplest way

In [ ]:
prompt = "Somatic hypermutation allows the immune system to"
generator = pipeline("text-generation", model=f"{user_name}/{ckpt_name}")
generator(prompt)

### More complex | More flexible

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(f"{user_name}/{ckpt_name}")
inputs = tokenizer(prompt, return_tensors="pt").input_ids
model = AutoModelForCausalLM.from_pretrained(f"{user_name}/{ckpt_name}")
outputs = model.generate(inputs, max_new_tokens=100, do_sample=True, top_k=50, top_p=0.95)
tokenizer.batch_decode(outputs, skip_special_tokens=True)

For more details about the different text generation strategies and parameters for controlling generation, check out the [Text generation strategies](https://huggingface.co/docs/transformers/main/en/tasks/../generation_strategies) page.